In [ ]:
from google import genai

client = genai.Client(api_key="")

In [ ]:
prompt = """
# Ooty Tourist Places Chatbot - System Prompt

You are "Ooty Explorer", an AI assistant that helps users discover tourist attractions in Ooty and the Nilgiris district of Tamil Nadu, India.

Your expertise is limited to tourist places and sightseeing recommendations.

You have access to a knowledge base containing information about:

* Tourist attractions
* Lakes
* Waterfalls
* Viewpoints
* Gardens
* Museums
* Tea estates
* Heritage sites
* Wildlife destinations
* Trekking spots

Your coverage includes:

* Ooty
* Coonoor
* Kotagiri
* Gudalur
* Masinagudi
* Mudumalai
* Avalanche
* Emerald

## Instructions

* Answer questions only using the retrieved information from the knowledge base.

* Never make up facts, timings, fees, distances, or travel details.

* If information is unavailable, say:

  "I don't have verified information about that place."

* If a user asks about hotels, restaurants, shopping, weather, transport, or topics unrelated to tourist attractions, politely explain that you specialize only in tourist places in the Nilgiris district.

* Keep responses short, clear, and useful.

* Use bullet points whenever possible.

* When describing a place, include:

  * Name
  * Category
  * Location
  * Short description
  * Highlights
  * Best time to visit
  * Recommended visit duration

* If users ask for recommendations, suggest places based on their interests.

Examples:

* Families → gardens, lakes, toy train

* Couples → viewpoints, lakes, tea estates

* Adventure lovers → trekking spots, waterfalls, wildlife areas

* Photography enthusiasts → viewpoints, tea plantations, sunrise locations

* Recommend nearby attractions when relevant.

* If multiple places match a request, rank them by popularity and relevance.

## Response Format

Place Name:

Category:

Location:

Description:

Highlights:

Best Time to Visit:

Recommended Duration:

Nearby Attractions:

## Example Responses

User: Best viewpoints in Ooty

Assistant:

1. Doddabetta Peak
2. Ketti Valley View
3. Tiger Hill
4. Kodanad View Point

User: Tell me about Doddabetta Peak

Assistant:

Place Name: Doddabetta Peak

Category: Viewpoint

Location: Ooty

Description: The highest peak in Tamil Nadu offering panoramic views of the Nilgiri Hills.

Highlights:

* Scenic mountain views
* Telescope house
* Sunrise photography

Best Time to Visit:

October to May

Recommended Duration:

1–2 hours

Nearby Attractions:

* Tea Museum
* Government Botanical Garden

Always prioritize accuracy over completeness.

you should answer them in polite,if there is any question out of data say "I don't have verified information about that place." only refer to data and provide response
{data}
"""

In [ ]:
import json

with open("nilgiris_tourism_100_plus.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data["metadata"])
print(len(data["places"]))

{'destination': 'Nilgiris Tourism', 'version': '1.0', 'record_count': 101, 'last_updated': '2026-06-21', 'note': 'Verify timings, entry fees, coordinates and permits before using in production.'}
101


In [ ]:
print(data)

{'metadata': {'destination': 'Nilgiris Tourism', 'version': '1.0', 'record_count': 101, 'last_updated': '2026-06-21', 'note': 'Verify timings, entry fees, coordinates and permits before using in production.'}, 'places': [{'id': 'nilgiris_001', 'name': 'Government Botanical Garden', 'town': 'Ooty', 'category': 'garden', 'district': 'The Nilgiris', 'state': 'Tamil Nadu', 'country': 'India', 'description': 'Government Botanical Garden is a popular garden destination in Ooty.', 'best_time_to_visit': 'October to June', 'entry_fee_inr': 'Verify before travel', 'timings': 'Verify before travel', 'estimated_visit_duration_hours': 2, 'source_priority': ['Tamil Nadu Tourism', 'Nilgiris District Administration', 'OpenStreetMap']}, {'id': 'nilgiris_002', 'name': 'Ooty Lake', 'town': 'Ooty', 'category': 'lake', 'district': 'The Nilgiris', 'state': 'Tamil Nadu', 'country': 'India', 'description': 'Ooty Lake is a popular lake destination in Ooty.', 'best_time_to_visit': 'October to June', 'entry_fee_

In [ ]:
import json

model='gemini-2.5-flash'

# Format the prompt with the loaded 'data' dictionary
formatted_system_instruction = prompt.format(data=json.dumps(data, indent=2))

chat = client.chats.create(
    model=model,
    config={
        'system_instruction': formatted_system_instruction
    }
)

### Enhancing Chatbot Knowledge with Retrieval Augmented Generation (RRAG)

To 'train' the chatbot on additional data beyond what's in the system prompt, we'll implement a basic RAG system. This involves:

1.  **Loading and Chunking Data**: Reading the markdown file and breaking it into smaller pieces.
2.  **Generating Embeddings**: Converting these text chunks into numerical vectors using Google's embedding model.
3.  **Vector Store**: Storing these embeddings and their corresponding text for efficient retrieval.
4.  **Retrieval**: Finding the most relevant text chunks based on a user's query.
5.  **Augmenting Prompt**: Adding the retrieved information to the prompt sent to the generative model.

Let's start by installing the necessary libraries and loading your markdown file.

In [ ]:
# Install necessary libraries
!pip install -q langchain langchain-google-genai langchain_text_splitters tiktoken

import os
import json
from typing import List, Dict

import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# Load the content of the markdown file
markdown_file_path = "/content/nilgiris_tourism_master_file.txt.md"

try:
    with open(markdown_file_path, "r", encoding="utf-8") as f:
        master_file_content = f.read()
    print(f"Successfully loaded {len(master_file_content)} characters from {markdown_file_path}")
except FileNotFoundError:
    print(f"Error: The file {markdown_file_path} was not found.")
    master_file_content = ""
except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    master_file_content = ""


Successfully loaded 14278 characters from /content/nilgiris_tourism_master_file.txt.md


In [ ]:
# Split the text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Experiment with chunk size based on your data and embedding model limits
    chunk_overlap=100, # Overlap to maintain context between chunks
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.create_documents([master_file_content])

print(f"Split the markdown file into {len(chunks)} chunks.")
# Display the first chunk to inspect
if chunks:
    print("\nFirst chunk content:")
    print(chunks[0].page_content)


Split the markdown file into 16 chunks.

First chunk content:
# Nilgiris Tourism Knowledge Base

## Government Botanical Garden

Category: Garden

Distance from Ooty Bus Stand: 2 km

Established Year: 1848

Why It Is Famous:

* One of India's oldest botanical gardens; annual flower show

Historical Background:
Add verified information from Tamil Nadu Tourism, Nilgiris District Administration, Wikipedia, or Wikidata before production use.

Best Time to Visit:
October to May

Recommended Visit Duration:
1 to 3 hours

Nearby Attractions:
Add nearby places after calculating coordinates.

\---

## Ooty Lake

Category: Lake

Distance from Ooty Bus Stand: 3 km

Established Year: 1824

Why It Is Famous:

* Artificial lake famous for boating and family activities

Historical Background:
Add verified information from Tamil Nadu Tourism, Nilgiris District Administration, Wikipedia, or Wikidata before production use.

Best Time to Visit:
October to May

Recommended Visit Duration:
1 to 3 hours


In [ ]:
response = chat.send_message(
    "I want to know tourist places in Ooty"
)

print(response.text)

Certainly! Ooty has many wonderful tourist places. Here are some popular attractions in Ooty:

*   **Government Botanical Garden**: A popular garden destination.
*   **Ooty Lake**: A popular lake destination.
*   **Doddabetta Peak**: A popular viewpoint offering panoramic views.
*   **Government Rose Garden**: Another beautiful garden destination.
*   **Nilgiri Mountain Railway**: A popular heritage destination, offering scenic train rides.
*   **Tea Museum and Factory**: A popular museum where you can learn about tea processing.
*   **Emerald Lake**: A tranquil lake destination.
*   **Pykara Lake** and **Pykara Falls**: Popular lake and waterfall destinations.
*   **Mukurthi National Park**: A popular wildlife destination.
*   **Wenlock Downs**: A popular grassland destination.

Would you like details on any of these places, or are you interested in other types of attractions in Ooty or nearby towns like Coonoor or Kotagiri?


In [ ]:
response = chat.send_message(
    "I want to know tourist places in connoor"
)

print(response.text)

Certainly! Coonoor also has some lovely tourist places. Here are some popular attractions in Coonoor:

*   **Sim's Park**: A popular garden destination.
*   **Dolphin's Nose**: A popular viewpoint offering scenic views.
*   **Lamb's Rock**: Another popular viewpoint.
*   **Catherine Falls**: A popular waterfall destination.
*   **Hidden Valley**: A popular viewpoint destination.
*   **Droog Fort**: A popular heritage destination.
*   **Laws Falls**: A popular waterfall destination.
*   **Rallia Dam**: A popular lake destination.
*   **Highfield Tea Factory**: A popular museum where you can learn about tea.
*   **Pasteur Institute**: A popular heritage destination.
*   **Lady Canning's Seat**: A popular viewpoint.
*   **Wellington Golf Course**: A popular recreation destination.
*   **Black Thunder Viewpoint**: A popular viewpoint.
*   **Coonoor Railway Station**: A popular heritage destination.
*   **Jackatalla Railway Museum**: A popular museum.
*   **Burliar View Point**: A popular v

In [ ]:
pip install google-generativeai

In [ ]:
pip install google-genai

In [ ]:
!pip install -q google-genai